## **Triple-pendulum model by PySINDy**

## **I. Setup and Preprocessing**

In [1]:
import os
import sys
sys.path.append("../utilities")

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.signal import savgol_filter
from sklearn.metrics import r2_score, mean_squared_error

import pysindy as ps

MY_DATA_TYPE = 'float32'
SEED = 42
np.random.seed(SEED)

opt_folder = 'output'
os.makedirs(opt_folder, exist_ok=True)

print(f"PySINDy version: {ps.__version__}")
print("Setup complete!")

PySINDy version: 2.0.0
Setup complete!


### **1.1 Load Data**

In [2]:
def load_triple_pendulum_data(mat_path, downsample=1):
    """Load triple pendulum data from .mat file."""
    data = loadmat(mat_path)
    theta1 = data['Theta1'][::downsample]
    theta2 = data['Theta2'][::downsample]
    theta3 = data['Theta3'][::downsample]
    d_theta1 = data['dTheta1'][::downsample]
    d_theta2 = data['dTheta2'][::downsample]
    d_theta3 = data['dTheta3'][::downsample]
    
    my_data = np.concatenate([theta1, theta2, theta3, d_theta1, d_theta2, d_theta3], axis=1)
    d_theta = np.concatenate([d_theta1, d_theta2, d_theta3], axis=1)
    
    return my_data, d_theta

# Load training data
data_1, d_theta_1 = load_triple_pendulum_data('TriplePendulum_Data/TripleDataFreeSwing_1_Dt_0_0001.mat', downsample=100)
data_2, d_theta_2 = load_triple_pendulum_data('TriplePendulum_Data/TripleDataFreeSwing_2_Dt_0_0001.mat', downsample=100)

my_data = np.concatenate([data_1, data_2], axis=0)
dt = 0.0001 * 100  # Time step

# Calculate second derivatives using Savitzky-Golay filter
window_length = 7
polyorder = 2
dd_theta_1 = savgol_filter(d_theta_1, window_length, polyorder, delta=dt, deriv=1, axis=0)
dd_theta_2 = savgol_filter(d_theta_2, window_length, polyorder, delta=dt, deriv=1, axis=0)

derivative_data_1 = np.concatenate([d_theta_1, dd_theta_1], axis=1)
derivative_data_2 = np.concatenate([d_theta_2, dd_theta_2], axis=1)
derivative_data = np.concatenate([derivative_data_1, derivative_data_2], axis=0)

print(f"Data shape: {my_data.shape}")
print(f"Derivative data shape: {derivative_data.shape}")

Data shape: (12002, 6)
Derivative data shape: (12002, 6)


### **1.2 Build Candidate Function Library**

In [ ]:
poly_library = ps.PolynomialLibrary(degree=1, include_bias=True)

fourier_library = ps.FourierLibrary(n_frequencies=2)  # n_frequencies=2 gives sin(x), cos(x), sin(2x), cos(2x)

library_1st = poly_library + fourier_library

# create 2nd order by tensor product of two 1st order
combined_library = ps.GeneralizedLibrary(
    [library_1st, library_1st],  
    tensor_array=[[1,1]]
)

# create 3rd order by tensor product of 2nd order with 1st order
combined_library = ps.GeneralizedLibrary(
    [library_1st, combined_library],  
    tensor_array=[[1,1]]
)


In [10]:
# Create PySINDy model with STLSQ optimizer
model = ps.SINDy(
    feature_library=combined_library,
    optimizer=ps.STLSQ(threshold=0.005, alpha=0.0, max_iter=20)
)

# Fit the model
print("Training SINDy model...")
model.fit(my_data, t=dt, x_dot=derivative_data)

print("\nModel training complete!")
print("\nIdentified equations:")
model.print()

Training SINDy model...


/opt/anaconda3/envs/sreinet/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/anaconda3/envs/sreinet/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/anaconda3/envs/sreinet/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/anaconda3/envs/sreinet/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/anaconda3/envs/sreinet/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warni


Model training complete!

Identified equations:
(x0)' = 0.053 x0 x3 + 0.053 x3 x0 + 0.053 x0 x3 + 0.053 x0 x3 + 0.010 x0 x3 sin(1 x0) + -0.002 x0 x3 sin(1 x1) + 0.010 x0 sin(1 x0) x3 + -0.002 x0 sin(1 x1) x3 + 0.007 x1 x3 sin(1 x0) + 0.002 x1 x3 sin(1 x1) + 0.007 x1 sin(1 x0) x3 + 0.002 x1 sin(1 x1) x3 + 0.053 x3 x0 + 0.053 x3 x0 + 0.010 x3 x0 sin(1 x0) + -0.002 x3 x0 sin(1 x1) + 0.007 x3 x1 sin(1 x0) + 0.002 x3 x1 sin(1 x1) + 0.010 x3 sin(1 x0) x0 + 0.007 x3 sin(1 x0) x1 + -0.002 x3 sin(1 x1) x0 + 0.002 x3 sin(1 x1) x1 + 0.010 sin(1 x0) x0 x3 + 0.007 sin(1 x0) x1 x3 + 0.010 sin(1 x0) x3 x0 + 0.007 sin(1 x0) x3 x1 + -0.002 sin(1 x1) x0 x3 + 0.002 sin(1 x1) x1 x3 + -0.002 sin(1 x1) x3 x0 + 0.002 sin(1 x1) x3 x1
(x1)' = 0.081 x0 x4 + 0.081 x4 x0 + 0.081 x0 x4 + 0.081 x0 x4 + -0.020 x0 x0 x4 + 0.001 x0 x1 x4 + -0.020 x0 x4 x0 + 0.001 x0 x4 x1 + -0.001 x0 x4 sin(1 x0) + -0.001 x0 x4 sin(1 x1) + -0.001 x0 sin(1 x0) x4 + -0.001 x0 sin(1 x1) x4 + 0.001 x1 x0 x4 + 0.001 x1 x4 x0 + 0.007 x1 x4

### **3.1 Load Validation Data and Simulate**

In [ ]:
# Load validation data
val_data, val_d_theta = load_triple_pendulum_data(
    'TriplePendulum_Data/TripleDataFreeSwing_3_Dt_0_0001.mat', 
    downsample=100
)

# Simulation parameters
t_test = np.arange(0, 0.2, 0.01)
initial_condition = val_data[0]

# Simulate 
x_sim = model.simulate(initial_condition, t_test)

print(f"Simulation complete! Shape: {x_sim.shape}")

### **3.2 Plot Results - Positions**

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8))

for i in range(3):
    axes[i].plot(t_test, x_sim[:, i], label='PySINDy', linewidth=2)
    axes[i].plot(t_test[:len(val_data)], val_data[:len(t_test), i], '--', label='True', linewidth=2)
    axes[i].set_ylabel(f'θ{i+1}')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

axes[2].set_xlabel('Time (s)')
plt.suptitle('Triple Pendulum - Angular Positions (PySINDy)')
plt.tight_layout()
plt.show()

### **3.3 Plot Results - Velocities**

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8))

for i in range(3):
    axes[i].plot(t_test, x_sim[:, i+3], label='PySINDy', linewidth=2)
    axes[i].plot(t_test[:len(val_data)], val_data[:len(t_test), i+3], '--', label='True', linewidth=2)
    axes[i].set_ylabel(f'dθ{i+1}/dt')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

axes[2].set_xlabel('Time (s)')
plt.suptitle('Triple Pendulum - Angular Velocities (PySINDy)')
plt.tight_layout()
plt.show()

### **3.4 Model Score**

In [ ]:
# Calculate model score on training data (PySINDy doesn't have validation score built-in)
train_score = model.score(my_data, t=dt, x_dot=derivative_data)
print(f"Model score on training data: {train_score:.6f}")